# A3C sur HalfCheetah : paralleliser l'actor-critic

**Algorithme** : A3C (Asynchronous Advantage Actor-Critic), extension parallele de A2C-GAE  
**Environnement** : HalfCheetah-v5 (MuJoCo, Gymnasium)  
**Papier** : [Mnih et al., 2016](https://arxiv.org/abs/1602.01783)

| Propriete | A2C | A3C |
|-----------|-----|-----|
| Model-free | oui | oui |
| On-policy | oui | oui |
| Actor-Critic | oui | oui |
| GAE | oui | oui |
| Replay buffer | non | non |
| Asynchrone | non | oui |
| Multi-worker | non | oui |
| Shared memory | non | oui |

**Pre-requis** : notebook 04 (A2C et A2C-GAE sur HalfCheetah). On reimplemente toutes les briques ici, mais on explique brievement ce qui a deja ete couvert en profondeur dans le notebook 04.

Dans le notebook 04, on a construit A2C et A2C-GAE : un seul agent, un seul environnement, une boucle sequentielle de collecte et de mise a jour. Ce notebook franchit une etape supplementaire : **paralleliser** l'entrainement sur plusieurs workers independants, chacun avec son propre environnement, tous contribuant a un modele partage unique.

## Pourquoi paralleliser ?

### La limite d'A2C sequentiel

A2C avec un seul environnement a deux goulots d'etranglement :

1. **Lenteur de collecte** : le GPU attend que le CPU simule l'environnement. La mise a jour attend la fin de la collecte. Ces deux phases sont strictement sequentielles.
2. **Correlation temporelle des donnees** : un seul agent dans un seul environnement genere des transitions fortement correlees dans le temps. $s_t$ et $s_{t+1}$ sont proches dans l'espace des etats. Le signal de gradient est donc potentiellement biaise vers une region de l'espace.

### La solution A3C : N workers, 1 modele partage

On lance $N$ workers en **parallele**, chacun avec son propre environnement et sa propre copie locale du modele :

$$\underbrace{\text{Shared Model}}_{\theta, \phi} \xleftarrow{\text{gradients}} \begin{cases} \text{Worker 1 : env}_1, \text{model local}_1 \\ \text{Worker 2 : env}_2, \text{model local}_2 \\ \vdots \\ \text{Worker N : env}_N, \text{model local}_N \end{cases}$$

**Deux avantages :**

1. **Plus de donnees par unite de temps** : $N$ environments simultes en parallele $\approx N$ fois plus de transitions par seconde wall-clock (sur des machines multi-coeurs).
2. **Decorrelation des donnees** : chaque worker explore une partie differente de l'espace des etats (etats initiaux differents, historiques differents). Les gradients provenant de regions differentes de l'espace des etats se **moyennent** naturellement, ce qui remplace le replay buffer des methodes off-policy.

**Analogie.** A2C, c'est un seul eleve qui fait ses devoirs en faisant les exercices l'un apres l'autre. A3C, c'est une classe entiere : chaque eleve travaille sur un probleme different, et le professeur (modele partage) apprend de tous simultanement. La diversite des problemes traites accelere l'apprentissage global.

### Ce qui ne change pas

La **loss**, les **avantages GAE**, la **politique gaussienne**, le **critic** : rigoureusement identiques au notebook 04. A3C n'est pas un nouvel algorithme d'optimisation. C'est une **orchestration parallele** du meme algorithme A2C-GAE.

## Architecture A3C : le protocole worker

### Vue d'ensemble

```
PROCESSUS PRINCIPAL
├── shared_actor   (poids en shared memory)
├── shared_critic  (poids en shared memory)
├── shared_optimizer (etat Adam en shared memory)
└── mp.Queue (pour recevoir les metriques des workers)

WORKER 1 (mp.Process)          WORKER 2 (mp.Process)       ...
├── local_env_1                ├── local_env_2
├── local_actor_1 (copie)      ├── local_actor_2 (copie)
├── local_critic_1 (copie)     ├── local_critic_2 (copie)
└── boucle :                   └── boucle :
    1. sync local <- shared        1. sync local <- shared
    2. collecter t_max steps       2. collecter t_max steps
    3. calculer gradients          3. calculer gradients
    4. pousser grads -> shared     4. pousser grads -> shared
    5. shared_optimizer.step()     5. shared_optimizer.step()
```

### Les 4 etapes d'un cycle worker

| Etape | Operation | Code |
|-------|-----------|------|
| **Sync** | Copier les poids du modele partage vers le modele local | `local_model.load_state_dict(shared_model.state_dict())` |
| **Collect** | Jouer $t_{max}$ pas dans l'environnement local | boucle `env.step()` |
| **Backward** | Calculer les gradients sur le modele local | `loss.backward()` |
| **Push** | Copier les gradients locaux vers le modele partage, appeler optimizer | `ensure_shared_grads()` + `shared_opt.step()` |

### Pourquoi les gradients ne necessitent pas de verrou ?

Plusieurs workers peuvent appeler `shared_optimizer.step()` en meme temps, avec des gradients calcules sur des poids potentiellement differents (chaque worker a sync au debut de son cycle, mais d'autres workers ont pu mettre a jour le modele partage entre-temps). Ces gradients sont dits **"stale"** (rassis).

C'est intentionnel et ca marche : c'est le principe **Hogwild!** ([Recht et al., 2011](https://arxiv.org/abs/1106.5730)). La SGD stochastique est robuste au bruit et aux mises a jour desordonnnees tant que le taux d'apprentissage est suffisamment petit. En pratique, les conflits d'ecriture sur les poids du modele partage sont rares et ont peu d'impact sur la convergence.

### Flux complet d'un cycle asynchrone

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    shared["A3CAgent<br/>acteur + critique + SharedAdam"]
    local["copie locale du worker"]
    env["environnement propre au worker"]
    rollout["rollout de t_max pas"]
    grad["backward local"]
    push["copie des gradients"]
    shared -->|"synchronisation des poids"| local
    local --> env --> rollout --> grad --> push --> shared
    class shared primary
    class rollout,grad,push secondary
```

Chaque worker possède son environnement et son graphe de calcul. Seuls les paramètres globaux et
l'état d'Adam sont partagés. Les trajectoires ne sont donc jamais mélangées dans un buffer central :
elles produisent des gradients locaux, appliqués de façon asynchrone au même agent partagé.

In [ ]:
import importlib
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
from IPython.display import Video, display
import torch.multiprocessing as mp
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import time
from pathlib import Path

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100
print(f"PyTorch : {torch.__version__}")
print(f"Gymnasium : {gym.__version__}")

# multiprocessing : methode 'spawn' obligatoire sur macOS et recommandee en general
# force=True pour etre idempotent (permet de reexecuter la cellule)
mp.set_start_method("spawn", force=True)
print(f"mp start method : {mp.get_start_method()}")

# Verifier MuJoCo
try:
    _env_test = gym.make("HalfCheetah-v5")
    _env_test.close()
    print("MuJoCo : disponible")
except Exception as e:
    print(f"MuJoCo : ERREUR -> {e}")
    print("Installez avec : pip install gymnasium[mujoco]")

## Shared Memory et torch.multiprocessing

### Le probleme du GIL Python

Python dispose d'un **GIL** (Global Interpreter Lock) : un seul thread Python peut executer du bytecode Python a la fois. Pour du vrai parallelisme CPU, il faut des **processus** (`mp.Process`) et non des threads (`threading.Thread`). Chaque processus a son propre interprete Python et son propre espace memoire.

Mais comment les workers partagent-ils le modele si chaque processus a sa propre memoire ?

### `model.share_memory()` : la cle

PyTorch permet de placer les tenseurs d'un modele dans la **memoire partagee inter-processus** (`POSIX shared memory` ou `mmap`). Apres `model.share_memory()`, les tenseurs du modele sont accessibles en lecture et en ecriture depuis n'importe quel processus fils, sans copie :

```python
shared_actor = GaussianActor(obs_dim, action_dim)
shared_actor.share_memory()  # les poids sont maintenant dans shared memory

# Dans chaque worker (processus fils) :
# shared_actor.parameters() pointe vers la MEME memoire physique
```

### Compteurs atomiques et file de resultats

Pour coordonner les workers :

| Outil | Usage | Exemple |
|-------|-------|---------|
| `mp.Value('i', 0)` | Compteur entier atomique (avec lock) | `global_step`, `global_ep` |
| `mp.Queue()` | File FIFO inter-processus | Envoyer metriques du worker au processus principal |

### La contrainte `spawn` sur macOS

Sur macOS (et Windows), la methode par defaut est `spawn` : chaque processus fils est cree proprement (pas de `fork`). Cela signifie que les fonctions passees a `mp.Process` doivent etre **picklables** (serialisables). En pratique : pas de lambdas, pas de fonctions imbriquees dans d'autres fonctions, pas de fermetures sur des objets non picklables. On utilise des fonctions de haut niveau definies au niveau du module.

### Note sur l'exception d'import A3C

A3C garde volontairement un module généré (`_a3c_components.py`) puis un import plus bas : avec `multiprocessing` en mode `spawn`, les workers doivent réimporter un module réel, pas des définitions restées dans `__main__` du notebook.


In [ ]:
%%writefile _a3c_components.py
"""Components A3C ecrits dans un fichier pour compatibilite spawn.

Python multiprocessing avec spawn re-importe __main__ dans les subprocesses.
Dans un notebook Jupyter, __main__ est le kernel et les fonctions des cellules
n'y existent pas. Ce fichier rend tout importable par les workers.
"""
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.multiprocessing as mp
import numpy as np
import gymnasium as gym
from torch.distributions import Normal


def orthogonal_init(module, gain=1.0):
    """Initialisation orthogonale recommandee pour les reseaux actor-critic."""
    if isinstance(module, nn.Linear):
        nn.init.orthogonal_(module.weight, gain=gain)
        nn.init.zeros_(module.bias)


class GaussianActor(nn.Module):
    """Politique gaussienne pi(a|s; theta) pour actions continues.
    
    Identique au notebook 04 : 2 couches de 256 neurones, Tanh,
    init orthogonale, log_std comme parametre global appris.
    """

    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

        self.apply(lambda m: orthogonal_init(m, gain=np.sqrt(2)))
        orthogonal_init(self.mu_head, gain=0.01)

    def forward(self, x: torch.Tensor):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        mu = self.mu_head(x)
        std = self.log_std.exp().expand_as(mu)
        return Normal(mu, std)

    def get_action(self, obs: np.ndarray):
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        dist = self.forward(obs_t)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        return action.squeeze(0).numpy(), log_prob.squeeze(0)

    def evaluate(self, obs_t: torch.Tensor, actions_t: torch.Tensor):
        dist = self.forward(obs_t)
        log_prob = dist.log_prob(actions_t).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return log_prob, entropy


class CriticNetwork(nn.Module):
    """Reseau de valeur V(s; phi) : observation -> scalaire.
    
    Identique au notebook 04.
    """

    def __init__(self, obs_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

        self.apply(lambda m: orthogonal_init(m, gain=np.sqrt(2)))
        orthogonal_init(self.value_head, gain=1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.value_head(x).squeeze(-1)


# --- Demo ---
torch.manual_seed(42)
obs_dim, action_dim = 17, 6
actor_test = GaussianActor(obs_dim, action_dim)
critic_test = CriticNetwork(obs_dim)

print(f"GaussianActor : {sum(p.numel() for p in actor_test.parameters()):,} parametres")
print(f"CriticNetwork : {sum(p.numel() for p in critic_test.parameters()):,} parametres")

# Test share_memory
actor_test.share_memory()
critic_test.share_memory()
print(f"\nApres share_memory() :")
print(f"  actor.fc1.weight.is_shared() = {actor_test.fc1.weight.is_shared()}")
print(f"  critic.fc1.weight.is_shared() = {critic_test.fc1.weight.is_shared()}")
print("  -> Les poids sont accessibles depuis les processus fils")

## SharedAdam : pourquoi l'optimiseur standard ne suffit pas

### Le probleme de l'etat Adam

Adam maintient pour chaque parametre $\theta_i$ deux moments :
- $m_i$ : moyenne exponentielle des gradients ($\hat{\beta}_1$-EWMA)
- $v_i$ : variance exponentielle des gradients carre ($\hat{\beta}_2$-EWMA)

$$m_i \leftarrow \beta_1 m_i + (1-\beta_1) g_i, \quad v_i \leftarrow \beta_2 v_i + (1-\beta_2) g_i^2$$

$$\theta_i \leftarrow \theta_i - \alpha \frac{\hat{m}_i}{\sqrt{\hat{v}_i} + \epsilon}$$

Avec `torch.optim.Adam` standard, ces moments $m_i$ et $v_i$ sont stockes comme des tenseurs **locaux** (non partages). Quand un worker appelle `shared_optimizer.step()`, l'optimiseur ne met a jour que ses propres moments locaux, pas ceux des autres workers. Les moments ne sont pas synchronises entre workers.

### La solution : SharedAdam

On cree un `SharedAdam` qui place les moments $m$ et $v$ dans la **shared memory** des sa creation, via `share_memory_()`. Tous les workers partagent le meme etat Adam, ce qui ameliore la stabilite numerique de l'optimiseur a travers les mises a jour asynchrones.

In [ ]:
%%writefile -a _a3c_components.py


class SharedAdam(optim.Adam):
    """Adam dont l'etat (exp_avg, exp_avg_sq) est en shared memory.
    
    Necessite que les parametres du modele soient eux-memes en shared memory
    (model.share_memory() appele avant).
    """

    def __init__(self, params, lr=1e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        super().__init__(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        # Initialiser l'etat Adam et le placer en shared memory
        for group in self.param_groups:
            for p in group["params"]:
                # Creer l'etat si pas encore fait
                state = self.state[p]
                state["step"] = torch.zeros(1).share_memory_()
                state["exp_avg"] = torch.zeros_like(p.data).share_memory_()
                state["exp_avg_sq"] = torch.zeros_like(p.data).share_memory_()


# --- Demo ---
torch.manual_seed(42)
shared_actor_demo = GaussianActor(obs_dim, action_dim)
shared_critic_demo = CriticNetwork(obs_dim)
shared_actor_demo.share_memory()
shared_critic_demo.share_memory()

all_params = list(shared_actor_demo.parameters()) + list(shared_critic_demo.parameters())
shared_opt_demo = SharedAdam(all_params, lr=3e-4)

# Verifier que l'etat est bien en shared memory
first_param = list(shared_actor_demo.parameters())[0]
state = shared_opt_demo.state[first_param]
print("SharedAdam cree :")
print(f"  exp_avg.is_shared()    = {state['exp_avg'].is_shared()}")
print(f"  exp_avg_sq.is_shared() = {state['exp_avg_sq'].is_shared()}")
print(f"  step.is_shared()       = {state['step'].is_shared()}")
print("  -> L'etat Adam est accessible depuis tous les workers")


class A3CAgent:
    """Assemblage du modèle global et de l'optimiseur partagés."""

    def __init__(self, obs_dim, action_dim, hidden_dim, learning_rate):
        self.shared_actor = GaussianActor(obs_dim, action_dim, hidden_dim=hidden_dim)
        self.shared_critic = CriticNetwork(obs_dim, hidden_dim=hidden_dim)
        self.shared_actor.share_memory()
        self.shared_critic.share_memory()
        parameters = list(self.shared_actor.parameters()) + list(self.shared_critic.parameters())
        self.shared_optimizer = SharedAdam(parameters, lr=learning_rate)

    def worker_components(self):
        # Le worker reste une fonction top-level picklable pour multiprocessing.spawn.
        return self.shared_actor, self.shared_critic, self.shared_optimizer

    def models(self):
        return self.shared_actor, self.shared_critic


In [ ]:
%%writefile -a _a3c_components.py


def compute_gae(rewards: np.ndarray, dones: np.ndarray, values: np.ndarray,
                last_value: float, gamma: float = 0.99,
                gae_lambda: float = 0.95) -> tuple:
    """Calcule les avantages GAE et les returns bootstrappes.
    
    Identique au notebook 04 : GAE(gamma, lambda) via accumulation recursive
    des erreurs TD delta_t = r_t + gamma * V(s_{t+1}) - V(s_t).
    
    Args:
        rewards    : (t_max,) recompenses du rollout
        dones      : (t_max,) flags de fin d'episode
        values     : (t_max,) V(s_t) predit par le critic
        last_value : V(s_{t+n}) pour le bootstrap final
        gamma      : facteur d'actualisation
        gae_lambda : parametre GAE (0=TD(0), 1=Monte Carlo)
    
    Returns:
        advantages : (t_max,) avantages GAE
        returns    : (t_max,) cibles pour le critic
    """
    t_max = len(rewards)
    advantages = np.zeros(t_max, dtype=np.float32)
    last_gae = 0.0
    values_extended = np.append(values, last_value)

    for t in reversed(range(t_max)):
        non_terminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * values_extended[t + 1] * non_terminal - values_extended[t]
        last_gae = delta + gamma * gae_lambda * non_terminal * last_gae
        advantages[t] = last_gae

    returns = advantages + values
    return advantages, returns


# --- Demo rapide ---
np.random.seed(42)
rewards_demo = np.random.randn(20).astype(np.float32) * 0.5 + 0.3
dones_demo = np.zeros(20, dtype=np.float32)
dones_demo[9] = 1.0
values_demo = np.ones(20, dtype=np.float32) * 0.5

adv, ret = compute_gae(rewards_demo, dones_demo, values_demo, last_value=0.5)
print(f"compute_gae (identique notebook 04) :")
print(f"  rewards shape : {rewards_demo.shape}")
print(f"  advantages : mean={adv.mean():.4f}, std={adv.std():.4f}")
print(f"  returns    : mean={ret.mean():.4f}, std={ret.std():.4f}")
print(f"  Rupture correcte apres done au pas 9 : adv[9]={adv[9]:.4f} (isole)")

## Anatomie d'un worker A3C

### Pseudocode detaille

$$
\boxed{
\begin{aligned}
&\textbf{WORKER}\bigl(
\text{rank},\ \theta_A^{shared},\ \theta_V^{shared},\
\text{opt}^{shared},\ g_{step},\ g_{ep},\ Q_{result},\ \text{config}
\bigr)\\[2mm]
&\quad \text{Créer } env \leftarrow \mathrm{gym.make}(\text{config.env\_id})\\
&\quad \text{Créer } \theta_A^{local} \leftarrow \mathrm{GaussianActor},\qquad
        \theta_V^{local} \leftarrow \mathrm{CriticNetwork}\\
&\quad o_0 \leftarrow env.reset(\text{seed}=\text{rank})
\\[2mm]
&\quad \textbf{tant que } g_{step} < T_{\max}\ \textbf{faire}\\[1mm]
&\qquad \textcolor{gray}{\triangleright\ \text{1. Synchroniser les copies locales depuis les poids partagés}}\\
&\qquad \theta_A^{local} \leftarrow \theta_A^{shared},\qquad
        \theta_V^{local} \leftarrow \theta_V^{shared}
\\[1mm]
&\qquad \textcolor{gray}{\triangleright\ \text{2. Collecter } t_{\max} \text{ transitions dans l'environnement local}}\\
&\qquad \mathcal{B} \leftarrow \varnothing\\
&\qquad \textbf{pour } t=1,\dots,t_{\max}\ \textbf{faire}\\
&\qquad\qquad a_t,\ \log \pi(a_t\mid o_t)
        \leftarrow \pi_{\theta_A^{local}}(o_t)\\
&\qquad\qquad v_t \leftarrow V_{\theta_V^{local}}(o_t)\\
&\qquad\qquad o_{t+1},\ r_t,\ d_t \leftarrow env.step(a_t)\\
&\qquad\qquad \mathcal{B} \leftarrow
        \mathcal{B}\cup\{o_t,a_t,r_t,d_t,v_t,\log \pi(a_t\mid o_t)\}\\
&\qquad\qquad g_{step} \leftarrow g_{step}+1\\
&\qquad\qquad o_t \leftarrow
        \begin{cases}
        env.reset() & \text{si } d_t=1,\\
        o_{t+1} & \text{sinon.}
        \end{cases}
\\
&\qquad \textbf{fin pour}\\[1mm]
&\qquad v_{last} \leftarrow V_{\theta_V^{local}}(o_t)\,(1-d_t)
\\[1mm]
&\qquad \textcolor{gray}{\triangleright\ \text{3. Calculer la loss A2C/GAE sur les modèles locaux}}\\
&\qquad \hat A_t,\ \hat R_t \leftarrow
        \mathrm{GAE}(\mathcal{B}.r,\mathcal{B}.d,\mathcal{B}.v,\ v_{last})\\
&\qquad \hat A_t \leftarrow
        \frac{\hat A_t-\mu(\hat A)}{\sigma(\hat A)+\varepsilon}\\
&\qquad \mathcal{L}_{policy}
        \leftarrow -\mathbb{E}_t\bigl[\log \pi_{\theta_A^{local}}(a_t\mid o_t)\,\hat A_t\bigr]\\
&\qquad \mathcal{L}_{value}
        \leftarrow \mathbb{E}_t\bigl(V_{\theta_V^{local}}(o_t)-\hat R_t\bigr)^2\\
&\qquad \mathcal{L}
        \leftarrow \mathcal{L}_{policy}
        + c_v\,\mathcal{L}_{value}
        - c_e\,\mathcal{H}\bigl(\pi_{\theta_A^{local}}\bigr)\\
&\qquad \nabla_{\theta^{local}}\mathcal{L}
        \leftarrow \mathrm{backward}(\mathcal{L})
\\[1mm]
&\qquad \textcolor{gray}{\triangleright\ \text{4. Copier les gradients locaux vers les poids partagés}}\\
&\qquad \nabla\theta_A^{shared} \leftarrow \nabla\theta_A^{local},\qquad
        \nabla\theta_V^{shared} \leftarrow \nabla\theta_V^{local}\\
&\qquad \mathrm{clip\_grad\_norm}(\theta_A^{shared},\theta_V^{shared},\ 0.5)\\
&\qquad \mathrm{opt}^{shared}.\mathrm{step}()\\
&\qquad \mathrm{opt}^{shared}.\mathrm{zero\_grad}()
\\[1mm]
&\qquad Q_{result}.\mathrm{put}
        \bigl(\{\text{rank},\ \text{reward},\ \mathcal{L}_{policy},\ \mathcal{L}_{value},\dots\}\bigr)
\\[1mm]
&\quad \textbf{fin tant que}
\end{aligned}
}
$$

### Comparaison avec la boucle A2C

La boucle **interieure** (collect + backward + optimizer step) est identique a A2C. Ce qui s'ajoute :

- **Sync initial** (`load_state_dict`) : le worker commence avec les poids les plus recents du modele partage.
- **Push des gradients** (`ensure_shared_grads`) : les gradients calcules sur les poids locaux sont copies dans `.grad` des poids partages, afin que `shared_optimizer.step()` les applique.
- **Seed par worker** : `env.reset(seed=rank)` pour que chaque worker parte d'un etat different.

**Note.** `ensure_shared_grads` est necessaire parce que `loss.backward()` remplit `local_param.grad`. Mais `shared_optimizer.step()` lit `shared_param.grad`. On doit donc copier manuellement les gradients locaux vers les parametres partages.

In [ ]:
%%writefile -a _a3c_components.py


def ensure_shared_grads(local_model: nn.Module, shared_model: nn.Module):
    """Copie les gradients du modele local vers le modele partage.
    
    Apres loss.backward(), les gradients sont dans local_param.grad.
    shared_optimizer.step() lit shared_param.grad.
    On copie : local_param.grad -> shared_param.grad.
    """
    for local_param, shared_param in zip(local_model.parameters(),
                                         shared_model.parameters()):
        if local_param.grad is not None:
            shared_param.grad = local_param.grad.clone()


In [ ]:
%%writefile -a _a3c_components.py


def a3c_worker(
    rank: int,
    shared_actor: nn.Module,
    shared_critic: nn.Module,
    shared_optimizer: optim.Optimizer,
    global_step: mp.Value,
    global_ep: mp.Value,
    result_queue: mp.Queue,
    config: dict,
):
    """Fonction worker A3C - doit etre definie au niveau du module (picklable).
    
    Args:
        rank            : identifiant du worker (0, 1, ..., N-1)
        shared_actor    : acteur partage (shared memory)
        shared_critic   : critic partage (shared memory)
        shared_optimizer: optimiseur partage (SharedAdam)
        global_step     : compteur atomique du nombre total de pas
        global_ep       : compteur atomique du nombre total d'episodes
        result_queue    : file pour renvoyer les metriques au processus principal
        config          : dictionnaire de configuration
    """
    # --- Environnement local ---
    env_id = config["env_id"]
    env = gym.make(env_id)

    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    # --- Modeles locaux (copies independantes, non partagees) ---
    local_actor = GaussianActor(obs_dim, action_dim, hidden_dim=config["hidden_dim"])
    local_critic = CriticNetwork(obs_dim, hidden_dim=config["hidden_dim"])

    gamma = config["gamma"]
    gae_lambda = config["gae_lambda"]
    t_max = config["t_max"]
    value_coef = config["value_coef"]
    max_grad_norm = config["max_grad_norm"]
    total_timesteps = config["total_timesteps"]

    # Graine differente par worker pour la diversite des trajectoires
    obs, _ = env.reset(seed=rank * 1000 + 42)
    episode_reward = 0.0
    episode_rewards_local = []

    while True:
        # Verifier si on a depasse le budget total de steps
        with global_step.get_lock():
            if global_step.value >= total_timesteps:
                break

        # ================================================================
        # ETAPE 1 : SYNC local <- shared
        # On recupere les poids les plus recents du modele partage.
        # ================================================================
        local_actor.load_state_dict(shared_actor.state_dict())
        local_critic.load_state_dict(shared_critic.state_dict())
        local_actor.eval()
        local_critic.eval()

        # ================================================================
        # ETAPE 2 : COLLECT t_max steps
        # ================================================================
        observations = []
        actions = []
        rewards = []
        dones = []
        values = []
        log_probs = []

        for _ in range(t_max):
            with torch.no_grad():
                obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                policy_action, log_prob = local_actor.get_action(obs)
                value = local_critic(obs_t).item()

            # policy_action : action brute échantillonnée par la gaussienne.
            # env_action    : action clippée envoyée à HalfCheetah.
            # Le gradient doit rester cohérent avec policy_action, pas env_action.
            env_action = np.clip(policy_action, env.action_space.low, env.action_space.high)

            next_obs, reward, terminated, truncated, _ = env.step(env_action)
            done = terminated or truncated
            env_reward = float(reward)
            stored_reward = env_reward

            if truncated and not terminated:
                with torch.no_grad():
                    terminal_obs_t = torch.tensor(next_obs, dtype=torch.float32).unsqueeze(0)
                    terminal_value = local_critic(terminal_obs_t).item()
                stored_reward = env_reward + gamma * terminal_value

            observations.append(obs.copy())
            actions.append(policy_action.copy())  # action brute, cohérente avec log_prob
            rewards.append(float(stored_reward))
            dones.append(float(done))
            values.append(value)
            log_probs.append(log_prob.item())

            episode_reward += env_reward
            obs = next_obs

            with global_step.get_lock():
                global_step.value += 1

            if done:
                episode_rewards_local.append(episode_reward)
                with global_ep.get_lock():
                    global_ep.value += 1
                episode_reward = 0.0
                obs, _ = env.reset()

        # Bootstrap : valeur du dernier etat (0 si episode termine)
        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            last_val = local_critic(obs_t).item() * (1.0 - dones[-1])

        # ================================================================
        # ETAPE 3 : BACKWARD sur le modele local
        # Les gradients sont calcules sur les poids locaux.
        # ================================================================
        obs_arr = np.array(observations, dtype=np.float32)
        act_arr = np.array(actions, dtype=np.float32)
        rew_arr = np.array(rewards, dtype=np.float32)
        don_arr = np.array(dones, dtype=np.float32)
        val_arr = np.array(values, dtype=np.float32)

        # Avantages GAE et returns cibles (identique A2C-GAE)
        advantages_np, returns_np = compute_gae(
            rew_arr, don_arr, val_arr, last_val, gamma, gae_lambda
        )
        advantages_t = torch.tensor(advantages_np, dtype=torch.float32)
        returns_t = torch.tensor(returns_np, dtype=torch.float32)

        # Normalisation des avantages (stabilite)
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        obs_t = torch.tensor(obs_arr, dtype=torch.float32)
        act_t = torch.tensor(act_arr, dtype=torch.float32)

        # Evaluer avec les modeles LOCAUX (pour le graphe de calcul)
        local_actor.train()
        local_critic.train()
        log_probs_t, entropy_t = local_actor.evaluate(obs_t, act_t)
        values_pred_t = local_critic(obs_t)

        # Loss actor-critic identique a A2C-GAE
        policy_loss = -(log_probs_t * advantages_t).mean()
        value_loss = F.mse_loss(values_pred_t, returns_t)
        total_loss = policy_loss + value_coef * value_loss

        # Backprop sur les parametres locaux
        local_actor.zero_grad()
        local_critic.zero_grad()
        total_loss.backward()

        # ================================================================
        # ETAPE 4 : PUSH grads -> shared + optimizer step
        # On copie les gradients locaux vers les parametres partages,
        # puis on clippe et on applique le pas Adam.
        # ================================================================
        ensure_shared_grads(local_actor, shared_actor)
        ensure_shared_grads(local_critic, shared_critic)

        # Gradient clipping sur les parametres partages (essentiel MuJoCo)
        nn.utils.clip_grad_norm_(
            list(shared_actor.parameters()) + list(shared_critic.parameters()),
            max_grad_norm
        )
        shared_optimizer.step()
        shared_optimizer.zero_grad()

        # Envoyer les metriques au processus principal
        if episode_rewards_local:
            mean_rew = float(np.mean(episode_rewards_local))
            result_queue.put({
                "rank": rank,
                "mean_reward": mean_rew,
                "policy_loss": policy_loss.item(),
                "value_loss": value_loss.item(),
                "entropy": entropy_t.mean().item(),
                "global_step": global_step.value,
            })
            episode_rewards_local = []

    env.close()
    result_queue.put(None)  # signal de fin pour ce worker

In [ ]:
# Charger les composants depuis le fichier écrit plus haut.
# Le module doit exister sur disque pour que les subprocesses `spawn` puissent le réimporter.
a3c_components = importlib.import_module("_a3c_components")

orthogonal_init = a3c_components.orthogonal_init
GaussianActor = a3c_components.GaussianActor
CriticNetwork = a3c_components.CriticNetwork
SharedAdam = a3c_components.SharedAdam
A3CAgent = a3c_components.A3CAgent
compute_gae = a3c_components.compute_gae
ensure_shared_grads = a3c_components.ensure_shared_grads
a3c_worker = a3c_components.a3c_worker

print("Components chargés depuis _a3c_components.py (requis pour mp.spawn)")


In [ ]:
def train_a3c(config: dict) -> dict:
    """Lance l'entrainement A3C avec N workers.
    
    Args:
        config : dictionnaire de configuration (env_id, num_workers, total_timesteps, ...)
    
    Returns:
        history : dictionnaire avec les courbes d'entrainement
    """
    # methode spawn requise sur macOS (et recommandee partout)
    mp.set_start_method("spawn", force=True)

    # --- Creer et partager le modele global ---
    env_probe = gym.make(config["env_id"])
    obs_dim = env_probe.observation_space.shape[0]
    action_dim = env_probe.action_space.shape[0]
    env_probe.close()

    torch.manual_seed(config.get("seed", 42))
    agent = A3CAgent(
        obs_dim, action_dim,
        hidden_dim=config["hidden_dim"],
        learning_rate=config["lr"],
    )
    shared_actor, shared_critic, shared_optimizer = agent.worker_components()

    # --- Compteurs et file de resultats ---
    global_step = mp.Value("i", 0)   # int partage, thread-safe via lock
    global_ep = mp.Value("i", 0)     # nombre total d'episodes
    result_queue = mp.Queue()         # resultats envoyes par les workers

    # --- Lancer les workers ---
    num_workers = config["num_workers"]
    processes = []
    for rank in range(num_workers):
        p = mp.Process(
            target=a3c_worker,
            args=(
                rank, shared_actor, shared_critic, shared_optimizer,
                global_step, global_ep, result_queue, config
            ),
        )
        p.start()
        processes.append(p)

    # --- Collecter les resultats ---
    history = {"steps": [], "rewards": [], "policy_loss": [], "value_loss": [], "entropy": []}
    workers_done = 0
    log_interval = config.get("log_interval", 10_000)
    last_log_step = 0
    rewards_buffer = []

    print(f"A3C : {num_workers} workers, {config['total_timesteps']:,} steps total")
    print(f"Env : {config['env_id']}, t_max={config['t_max']}")
    print(f"{'=' * 60}")

    while workers_done < num_workers:
        result = result_queue.get()  # bloquant
        if result is None:
            workers_done += 1
            continue

        step = result["global_step"]
        rewards_buffer.append(result["mean_reward"])
        history["steps"].append(step)
        history["rewards"].append(result["mean_reward"])
        history["policy_loss"].append(result["policy_loss"])
        history["value_loss"].append(result["value_loss"])
        history["entropy"].append(result["entropy"])

        if step - last_log_step >= log_interval or step >= config["total_timesteps"]:
            last_log_step = step
            avg_rew = np.mean(rewards_buffer[-20:]) if rewards_buffer else 0.0
            print(
                f"Step {step:>7d} | "
                f"Worker {result['rank']} | "
                f"Reward={result['mean_reward']:>8.1f} | "
                f"Avg(20)={avg_rew:>8.1f} | "
                f"pi_loss={result['policy_loss']:>8.4f} | "
                f"v_loss={result['value_loss']:>8.4f}"
            )

    # Attendre la fin de tous les processus
    for p in processes:
        p.join()

    print(f"\nA3C termine. Episodes totaux : {global_ep.value}")
    return history, *agent.models()

## Subtilites de l'implementation

### `mp.set_start_method("spawn", force=True)`

Sur **macOS**, la methode `fork` est desactivee depuis Python 3.8 car elle provoque des deadlocks avec certaines bibliotheques (Objective-C runtime, MuJoCo). `spawn` cree un processus propre depuis zero. Le `force=True` permet de rappeler la fonction dans une cellule Jupyter sans lever d'exception si la methode est deja definie.

### Compteurs atomiques `mp.Value`

`mp.Value('i', 0)` cree un entier en memoire partagee avec un **verrou integre**. On l'utilise avec `with global_step.get_lock():` pour garantir que l'increment est atomique (pas de race condition entre workers).

### `mp.Queue` pour les metriques

Les workers ne peuvent pas ecrire directement dans des variables Python du processus principal (memoires separees). `mp.Queue` est une file FIFO thread-safe et process-safe. Chaque worker envoie ses metriques via `result_queue.put(dict)`. Le processus principal les lit via `result_queue.get()`.

### Gradient clipping **par worker, avant le push**

On clippe les gradients sur les parametres **partages** (apres `ensure_shared_grads`) avant d'appeler `shared_optimizer.step()`. Cela garantit que meme des gradients provenant de trajectoires extremes (grande recompense ou grande penalite) ne depassent pas `max_grad_norm=0.5`. Essentiel pour la stabilite sur MuJoCo.

### Pourquoi Hogwild! fonctionne

Plusieurs workers peuvent appeler `shared_optimizer.step()` simultanement. Les mises a jour sont desordonnees (asynchrones). Pourquoi ca converge quand meme ?

Le theoreme **Hogwild!** (Recht et al., 2011) montre que la SGD parallele sans verrou converge si :
- Le probleme est "creux" (les parametres mis a jour sont souvent differents entre workers) — approximativement vrai ici car chaque worker explore differentes regions de l'espace
- Le taux d'apprentissage est suffisamment petit

En pratique sur des problemes denses (tous les parametres sont mis a jour a chaque step), la convergence est legerement plus bruitee mais reste valide. L'asynchronisme ajoute du bruit equivalent a celui d'un taux d'apprentissage plus grand, ce qui peut meme ameliorer l'exploration.

$$\boxed{\text{A3C} = \text{A2C-GAE} + \text{Hogwild!} + \text{shared\_memory}}$$

In [ ]:
# --- Demo : 2 workers, 50k steps, Pendulum-v1 (leger) ---
# On utilise Pendulum-v1 pour la demo rapide car HalfCheetah demande plus de temps.
# La demo verifie que le protocole worker fonctionne correctement.

DEMO_CONFIG = {
    "env_id": "Pendulum-v1",
    "num_workers": 2,
    "total_timesteps": 50_000,
    "hidden_dim": 128,
    "lr": 3e-4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "t_max": 20,
    "value_coef": 0.5,
    "max_grad_norm": 0.5,
    "log_interval": 10_000,
    "seed": 42,
}

print("Demo A3C sur Pendulum-v1 (2 workers, 50k steps)...")
print("(Verifie que le protocole multiprocessing fonctionne)")
print()

t0 = time.time()
demo_history, demo_shared_actor, demo_shared_critic = train_a3c(DEMO_CONFIG)
t1 = time.time()

print(f"\nTemps ecoule : {t1 - t0:.1f}s")
if demo_history["rewards"]:
    print(f"Recompense finale (moy. 10 derniers) : "
          f"{np.mean(demo_history['rewards'][-10:]):.1f}")
print("\nDemo OK : les workers partagent bien le modele et l'optimiseur.")

## Hyperparametres de comparaison A2C vs A3C

On fixe les memes hyperparametres de base pour les deux afin d'isoler l'effet du parallelisme. La seule difference : A3C utilise `num_workers=4` et `t_max=20` au lieu d'un seul worker avec `n_steps=2048`.

| Parametre | A2C-GAE | A3C | Role |
|-----------|---------|-----|------|
| `hidden_dim` | 256 | 256 | Taille des couches cachees |
| `lr` | 3e-4 | 3e-4 | Taux d'apprentissage Adam |
| `gamma` | 0.99 | 0.99 | Facteur d'actualisation |
| `gae_lambda` | 0.95 | 0.95 | Parametre GAE |
| `value_coef` | 0.5 | 0.5 | Poids de la loss critic |
| `max_grad_norm` | 0.5 | 0.5 | Clipping du gradient |
| `total_timesteps` | 200 000 | 200 000 | Budget total de transitions |
| `seed` | 42 | 42 | Graine aleatoire |
| `n_steps` / `t_max` | 2048 | **20** | Longueur du rollout par worker |
| `num_workers` | 1 | **4** | Nombre de workers paralleles |

**Note sur `t_max=20` vs `n_steps=2048` :** Dans A3C, chaque worker travaille avec des rollouts courts (`t_max=20`) et met a jour frequemment. En synchrone (A2C), on utilise des rollouts longs (`n_steps=2048`) pour amortir le cout de la mise a jour. Avec 4 workers x 20 pas, A3C fait une mise a jour toutes les 20 transitions locales, soit 80 transitions wall-clock. A2C en fait une toutes les 2048. L'effet est similaire en nombre de transitions totales, mais A3C met a jour bien plus souvent, ce qui accelere l'adaptation.

In [ ]:
# === Reimplementation A2C-GAE pour la comparaison ===
# (Identique au notebook 04 - resumes brievement ici)

class RolloutBuffer:
    """Buffer on-policy pre-alloue (identique notebook 04)."""

    def __init__(self, n_steps, obs_dim, action_dim):
        self.n_steps = n_steps
        self.reset()
        self.obs_dim = obs_dim
        self.action_dim = action_dim

    def reset(self):
        self.observations = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.values = []
        self.ptr = 0

    def add(self, obs, action, reward, done, value):
        self.observations.append(obs.copy())
        self.actions.append(action.copy())
        self.rewards.append(float(reward))
        self.dones.append(float(done))
        self.values.append(float(value))
        self.ptr += 1

    @property
    def full(self):
        return self.ptr >= self.n_steps

    def get_arrays(self):
        return (
            np.array(self.observations, dtype=np.float32),
            np.array(self.actions, dtype=np.float32),
            np.array(self.rewards, dtype=np.float32),
            np.array(self.dones, dtype=np.float32),
            np.array(self.values, dtype=np.float32),
        )


In [ ]:
def train_a2c_gae(config: dict) -> dict:
    """Boucle d'entrainement A2C-GAE sequentiel (identique notebook 04)."""
    torch.manual_seed(config.get("seed", 42))
    np.random.seed(config.get("seed", 42))

    env = gym.make(config["env_id"])
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    actor = GaussianActor(obs_dim, action_dim, hidden_dim=config["hidden_dim"])
    critic = CriticNetwork(obs_dim, hidden_dim=config["hidden_dim"])
    optimizer = optim.Adam(
        list(actor.parameters()) + list(critic.parameters()),
        lr=config["lr"], eps=1e-5
    )

    n_steps = config["n_steps"]
    gamma = config["gamma"]
    gae_lambda = config["gae_lambda"]
    value_coef = config["value_coef"]
    max_grad_norm = config["max_grad_norm"]
    total_timesteps = config["total_timesteps"]
    log_interval = config.get("log_interval", 10_000)

    history = {"steps": [], "rewards": [], "policy_loss": [], "value_loss": []}
    buffer = RolloutBuffer(n_steps, obs_dim, action_dim)

    obs, _ = env.reset(seed=config.get("seed", 42))
    timesteps = 0
    episode_rewards = []
    current_ep_reward = 0.0
    last_log = 0

    print(f"A2C-GAE sequentiel : {total_timesteps:,} steps")
    print(f"{'=' * 60}")

    while timesteps < total_timesteps:
        buffer.reset()
        actor.eval()
        critic.eval()

        for _ in range(n_steps):
            with torch.no_grad():
                policy_action, _ = actor.get_action(obs)
                obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                value = critic(obs_t).item()

            # Même règle que pour A2C : on stocke l'action brute pour la loss,
            # on envoie seulement l'action clippée à l'environnement.
            env_action = np.clip(policy_action, env.action_space.low, env.action_space.high)

            next_obs, reward, terminated, truncated, _ = env.step(env_action)
            done = terminated or truncated
            env_reward = float(reward)
            stored_reward = env_reward

            if truncated and not terminated:
                with torch.no_grad():
                    terminal_obs_t = torch.tensor(next_obs, dtype=torch.float32).unsqueeze(0)
                    terminal_value = critic(terminal_obs_t).item()
                stored_reward = env_reward + gamma * terminal_value

            buffer.add(obs, policy_action, stored_reward, done, value)
            current_ep_reward += env_reward
            obs = next_obs
            timesteps += 1

            if done:
                episode_rewards.append(current_ep_reward)
                current_ep_reward = 0.0
                obs, _ = env.reset()
                
        # Bootstrap
        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            last_val = critic(obs_t).item()

        obs_arr, act_arr, rew_arr, don_arr, val_arr = buffer.get_arrays()
        advantages_np, returns_np = compute_gae(
            rew_arr, don_arr, val_arr, last_val, gamma, gae_lambda
        )

        obs_t2 = torch.tensor(obs_arr, dtype=torch.float32)
        act_t2 = torch.tensor(act_arr, dtype=torch.float32)
        advantages_t = torch.tensor(advantages_np, dtype=torch.float32)
        returns_t = torch.tensor(returns_np, dtype=torch.float32)
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        actor.train()
        critic.train()
        log_probs_t, _ = actor.evaluate(obs_t2, act_t2)
        values_pred_t = critic(obs_t2)

        policy_loss = -(log_probs_t * advantages_t).mean()
        value_loss = F.mse_loss(values_pred_t, returns_t)
        total_loss = policy_loss + value_coef * value_loss

        optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(
            list(actor.parameters()) + list(critic.parameters()), max_grad_norm
        )
        optimizer.step()

        if episode_rewards and (timesteps - last_log >= log_interval):
            last_log = timesteps
            mean_r = np.mean(episode_rewards[-10:])
            history["steps"].append(timesteps)
            history["rewards"].append(mean_r)
            history["policy_loss"].append(policy_loss.item())
            history["value_loss"].append(value_loss.item())
            print(
                f"Step {timesteps:>7d} | "
                f"Reward={mean_r:>8.1f} | "
                f"pi_loss={policy_loss.item():>8.4f} | "
                f"v_loss={value_loss.item():>8.4f}"
            )

    env.close()
    print(f"\nA2C-GAE termine.")
    return history, actor, critic


In [ ]:
# Hyperparametres communs
BASE_CONFIG = {
    "env_id": "HalfCheetah-v5",
    "hidden_dim": 256,
    "lr": 3e-4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "value_coef": 0.5,
    "max_grad_norm": 0.5,
    "total_timesteps": 200_000,
    "log_interval": 10_000,
    "seed": 42,
}

A2C_CONFIG = dict(BASE_CONFIG, n_steps=512)
A3C_CONFIG = dict(BASE_CONFIG, num_workers=4, t_max=20)

print("Configurations prets :")
print(f"  A2C-GAE : n_steps={A2C_CONFIG['n_steps']}, 1 worker")
print(f"  A3C     : t_max={A3C_CONFIG['t_max']}, {A3C_CONFIG['num_workers']} workers")


In [ ]:
# === Entrainement A2C-GAE ===
print("ENTRAINEMENT A2C-GAE (sequentiel)")
t_start_a2c = time.time()
history_a2c, actor_a2c, critic_a2c = train_a2c_gae(A2C_CONFIG)
t_end_a2c = time.time()
wall_clock_a2c = t_end_a2c - t_start_a2c
print(f"Temps wall-clock A2C-GAE : {wall_clock_a2c:.1f}s ({wall_clock_a2c/60:.1f} min)")

In [ ]:
# === Entrainement A3C ===
print("ENTRAINEMENT A3C (4 workers paralleles)")
t_start_a3c = time.time()
history_a3c, shared_actor_final, shared_critic_final = train_a3c(A3C_CONFIG)
t_end_a3c = time.time()
wall_clock_a3c = t_end_a3c - t_start_a3c
print(f"Temps wall-clock A3C : {wall_clock_a3c:.1f}s ({wall_clock_a3c/60:.1f} min)")

In [ ]:
# === Courbes comparatives A2C vs A3C ===

def moving_average(data, window=5):
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("A2C-GAE vs A3C sur HalfCheetah-v5 (200k steps)",
             fontsize=14, fontweight="bold")

color_a2c = "steelblue"
color_a3c = "coral"
window = 3

# --- Panel 1 : Recompenses en fonction des timesteps ---
ax = axes[0]
steps_a2c = np.array(history_a2c["steps"]) / 1000
steps_a3c = np.array(history_a3c["steps"]) / 1000

ax.plot(steps_a2c, history_a2c["rewards"], alpha=0.25, color=color_a2c)
ax.plot(steps_a3c, history_a3c["rewards"], alpha=0.25, color=color_a3c)

ma_a2c = moving_average(history_a2c["rewards"], window)
ma_a3c = moving_average(history_a3c["rewards"], window)
if len(steps_a2c) >= window:
    ax.plot(steps_a2c[window - 1:], ma_a2c, color=color_a2c, linewidth=2, label="A2C-GAE")
if len(steps_a3c) >= window:
    ax.plot(steps_a3c[window - 1:], ma_a3c, color=color_a3c, linewidth=2, label="A3C (4 workers)")

ax.set_xlabel("Timesteps (x1000)")
ax.set_ylabel("Recompense moyenne")
ax.set_title("Recompense vs Timesteps")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Panel 2 : Recompenses en fonction du temps wall-clock ---
ax = axes[1]
# Construire une echelle temporelle approximative
n_pts_a2c = len(history_a2c["steps"])
n_pts_a3c = len(history_a3c["steps"])
time_a2c = np.linspace(0, wall_clock_a2c / 60, n_pts_a2c)
time_a3c = np.linspace(0, wall_clock_a3c / 60, n_pts_a3c)

ax.plot(time_a2c, history_a2c["rewards"], alpha=0.25, color=color_a2c)
ax.plot(time_a3c, history_a3c["rewards"], alpha=0.25, color=color_a3c)

if len(time_a2c) >= window:
    ax.plot(time_a2c[window - 1:], ma_a2c, color=color_a2c, linewidth=2,
            label=f"A2C-GAE ({wall_clock_a2c/60:.1f} min)")
if len(time_a3c) >= window:
    ax.plot(time_a3c[window - 1:], ma_a3c, color=color_a3c, linewidth=2,
            label=f"A3C ({wall_clock_a3c/60:.1f} min)")

ax.set_xlabel("Temps wall-clock (min)")
ax.set_ylabel("Recompense moyenne")
ax.set_title("Recompense vs Temps reel")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Panel 3 : Value loss ---
ax = axes[2]
ax.plot(steps_a2c, history_a2c["value_loss"], alpha=0.25, color=color_a2c)
ax.plot(steps_a3c, history_a3c["value_loss"], alpha=0.25, color=color_a3c)

ma_vl_a2c = moving_average(history_a2c["value_loss"], window)
ma_vl_a3c = moving_average(history_a3c["value_loss"], window)
if len(steps_a2c) >= window:
    ax.plot(steps_a2c[window - 1:], ma_vl_a2c, color=color_a2c, linewidth=2, label="A2C-GAE")
if len(steps_a3c) >= window:
    ax.plot(steps_a3c[window - 1:], ma_vl_a3c, color=color_a3c, linewidth=2, label="A3C")

ax.set_xlabel("Timesteps (x1000)")
ax.set_ylabel("MSE")
ax.set_title("Value Loss (MSE critic)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resume
print("\nResume (20 derniers points) :")
print(f"{'Metrique':>22} | {'A2C-GAE':>12} | {'A3C':>12}")
print("-" * 52)
for key in ["rewards", "value_loss"]:
    v_a2c = np.mean(history_a2c[key][-20:]) if history_a2c[key] else float("nan")
    v_a3c = np.mean(history_a3c[key][-20:]) if history_a3c[key] else float("nan")
    print(f"{key:>22} | {v_a2c:>12.4f} | {v_a3c:>12.4f}")

print(f"{'wall_clock (min)':>22} | {wall_clock_a2c/60:>12.1f} | {wall_clock_a3c/60:>12.1f}")

In [ ]:
# === Démonstration finale : replay vidéo des politiques déterministes ===
def evaluate_and_display_agent(actor, label, n_episodes=3, seed=123, max_steps=1000, video_root="videos/05_a3c_halfcheetah"):
    """Évalue une politique déterministe et affiche le dernier replay vidéo enregistré."""
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(BASE_CONFIG["env_id"], render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    actor.eval()
    rewards = []

    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=seed + ep)
            total = 0.0
            steps = 0
            done = False
            while not done and steps < max_steps:
                with torch.no_grad():
                    obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                    action = actor(obs_t).mean.squeeze(0).numpy()
                    action = np.clip(action, env.action_space.low, env.action_space.high)
                obs, reward, terminated, truncated, _ = env.step(action)
                total += float(reward)
                steps += 1
                done = terminated or truncated
            rewards.append(total)
            print(f"[{label}] Épisode {ep + 1} : retour={total:+.1f} ({steps} pas)")
    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):+.1f} +/- {np.std(rewards):.1f}")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=640))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return rewards


demo_a2c_video = evaluate_and_display_agent(actor_a2c, "A2C-GAE")
demo_a3c_video = evaluate_and_display_agent(shared_actor_final, "A3C")


## Analyse : A2C vs A3C

### En termes de timesteps

A2C et A3C convergent vers des performances similaires quand on compare en nombre de transitions **totales** (axe horizontal du panel 1). C'est attendu : les deux algorithmes utilisent le meme estimateur GAE, la meme loss, les memes hyperparametres. La difference est dans l'orchestration, pas dans le calcul du gradient.

### En termes de temps wall-clock

A3C devrait etre plus rapide en **temps reel** (panel 2) car les 4 workers simulaent des environnements en parallele. Sur une machine avec suffisamment de coeurs CPU (HalfCheetah est entierement simule sur CPU par MuJoCo), le speedup peut etre proche de $N$ workers.

**Nuance.** Le overhead de la shared memory, des verrous (`mp.Value`), de la serialisation (`mp.Queue`) et du spawn des processus reduit le speedup theorique. Sur des machines avec peu de coeurs ou sur des environnements tres rapides a simuler, A2C avec des environments vectorises peut etre plus efficace.

### Variance inter-runs

A3C a une **variance plus elevee** entre differents runs : les mises a jour asynchrones introduisent un bruit supplementaire. Les courbes sont plus accidentees (Hogwild! = SGD bruise). A2C est plus reproductible.

### Tableau recapitulatif

| Dimension | A2C-GAE | A3C |
|-----------|---------|-----|
| Architecture | identique | identique |
| Estimateur d'avantage | GAE | GAE |
| Synchronicite | synchrone | asynchrone (Hogwild!) |
| Donnees par update | n_steps=2048 | t_max=20 x N workers |
| Decorrelation | faible (1 env) | forte (N envs differents) |
| Speedup wall-clock | reference | ~N workers (sature selon CPU) |
| Variance inter-runs | faible | plus elevee |
| Reproductibilite | bonne | plus difficile |
| Complexite code | simple | +multiprocessing |
| Usage moderne | environnements vectorises | moins courant (cf. note) |

**En pratique.** Dans la litterature moderne (post-2017), A3C est largement remplace par **PPO avec environnements vectorises** (`gym.vector.AsyncVectorEnv` ou `SyncVectorEnv`). Le parallelisme vectorise est plus simple a implementer, evite les problemes de shared memory, et donne des performances similaires. A3C reste important historiquement et pour les systemes distribues a grande echelle.

## Conclusion et ponts vers la suite

### Ce qu'on a construit

A3C est **la meme chose qu'A2C-GAE, parallelisee** :

- **GaussianActor** + **CriticNetwork** + **compute_gae** : identiques au notebook 04, reimplementes brievement ici.
- **SharedAdam** : Adam dont l'etat $(m, v)$ est en shared memory inter-processus.
- **Protocole worker** : sync $\to$ collect $\to$ backward $\to$ push, repetition jusqu'a convergence.
- **Hogwild!** : les mises a jour asynchrones sans verrou convergent grace a la robustesse de la SGD.

Les briques nouvelles introduites dans ce notebook :
- `model.share_memory()` : tenseurs PyTorch en memoire partagee inter-processus
- `mp.Process`, `mp.Value`, `mp.Queue` : orchestration multi-processus
- `ensure_shared_grads` : copie des gradients locaux vers le modele partage
- `SharedAdam` : optimiseur Adam avec etat partage

### Importance historique d'A3C

A3C (Mnih et al., 2016) a ete une **percee majeure** : le premier papier montrant qu'on peut entrainer des reseaux de neurones profonds en RL **sans replay buffer**, en parallelisant la collecte. Avant A3C, le replay buffer (DQN) semblait indispensable pour la stabilite. A3C a montre que la decorrelation temporelle peut aussi etre obtenue par diversite des trajectoires (N workers, N environments).

### Pont vers PPO

A2C et A3C font **un seul gradient step** par rollout. C'est peu efficace : on collecte `n_steps` transitions, on calcule un gradient, on jette les donnees. **PPO** (Proximal Policy Optimization, Schulman et al., 2017) fait $K$ epochs sur le meme rollout en protegeant contre les mises a jour trop grandes via le **clipping du ratio de probabilites** :

$$\mathcal{L}^{CLIP}(\theta) = \mathbb{E}_t\left[\min\left(\underbrace{r_t(\theta)}_{\pi_\theta(a_t|s_t)/\pi_{\theta_{old}}(a_t|s_t)} A_t,\ \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) A_t\right)\right]$$

Le clip empeche $r_t(\theta)$ de s'eloigner de 1 (c'est-a-dire empeche la nouvelle politique de trop diverger de l'ancienne). On peut faire plusieurs passes sur le meme rollout sans risque de mise a jour catastrophique. C'est pourquoi PPO est devenu le **standard de facto** du RL on-policy.

### Pont vers le parallelisme moderne

Le parallelisme d'A3C via `mp.Process` est aujourd'hui remplace par des approches plus simples et plus efficaces :
- **`gym.vector.AsyncVectorEnv`** : N environments en parallele dans un seul processus, sans shared memory ni gradients asynchrones. Plus simple, moins de bugs.
- **Systemes distribues** (RLlib, Acme, etc.) : parallelisme a grande echelle sur des clusters, avec des workers sur plusieurs machines.

$$\boxed{\text{REINFORCE} \to \text{A2C-GAE} \to \text{A3C} \to \text{PPO} \to \text{SAC/TD3}}$$

---

**References**
- Mnih, V. et al. (2016). Asynchronous Methods for Deep Reinforcement Learning. [arXiv:1602.01783](https://arxiv.org/abs/1602.01783). (A3C, et sa version synchrone A2C)
- Schulman, J. et al. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. [arXiv:1506.02438](https://arxiv.org/abs/1506.02438). (GAE)
- Schulman, J. et al. (2017). Proximal Policy Optimization Algorithms. [arXiv:1707.06347](https://arxiv.org/abs/1707.06347). (PPO)
- Recht, B. et al. (2011). Hogwild! A Lock-Free Approach to Parallelizing Stochastic Gradient Descent. [arXiv:1106.5730](https://arxiv.org/abs/1106.5730). (Convergence des SGD asynchrones)